In [1]:
# CTR scoring notebook 4: does the flag persist into the next month? (warehouse, two windows)
# Paste this whole file into ONE Colab cell and press Run. Needs the Colab secret HF_TOKEN (Notebook access ON).
# It reads only three monthly files, aggregates them per page in DuckDB, and prints NO ids and NO client info.

#### PART 1: load (needs internet + token)
!pip install -q duckdb huggingface_hub
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from scipy.stats import norm, poisson, spearmanr

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{userdata.get('HF_TOKEN')}')")
BASE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

PAIRS = [("2026-03", "2026-04"), ("2026-04", "2026-05")]   # (window A = used to flag, window B = next month, used to check)
MONTHS = sorted({m for p in PAIRS for m in p})

def load_month(m):
    q = f'''
    SELECT client_hash_id, content_hash_id,
           sum(gsc_impressions) AS impressions, sum(gsc_clicks) AS clicks,
           sum(gsc_avg_position * gsc_impressions) / sum(gsc_impressions) AS pos,
           sum(gsc_sum_position) * 1.0 / sum(gsc_impressions) AS pos_from_sum
    FROM read_parquet('{BASE}/month={m}/data_0.parquet')
    WHERE gsc_impressions > 0
    GROUP BY 1, 2
    '''
    return con.sql(q).df()

RAW = {}
for m in MONTHS:
    RAW[m] = load_month(m)
    print(m, "pages with impressions:", f"{len(RAW[m]):,}", "| clients:", RAW[m]["client_hash_id"].nunique())
#### END PART 1

#### PART 2: analysis (uses RAW only)
r = RAW[MONTHS[0]]
print("\nPosition scale check (median of pos, median of sum_position/impressions, median difference):",
      round(float(r["pos"].median()), 2), round(float(r["pos_from_sum"].median()), 2),
      round(float((r["pos"] - r["pos_from_sum"]).median()), 2))

BINS = [0, 1.5, 2.5, 3.5, 4.5, 5.5, 7.5, 10.5, 15.5, 20.5, 1000]
K = 50_000
MIN_IMP = 100

def score(raw):
    d = raw[(raw["impressions"] >= MIN_IMP) & raw["pos"].notna()].copy()
    d["ctr"] = d["clicks"] / d["impressions"]
    d["bucket"] = pd.cut(d["pos"], bins=BINS, labels=False)
    g = d.groupby("bucket")[["clicks", "impressions"]].sum()
    d["global_ctr"] = d["bucket"].map(g["clicks"] / g["impressions"])
    cb = d.groupby(["client_hash_id", "bucket"])[["clicks", "impressions"]].transform("sum")
    li, lc = cb["impressions"] - d["impressions"], cb["clicks"] - d["clicks"]
    loo = np.where(li > 0, lc / li.where(li > 0, 1), 0.0)
    w = li / (li + K)
    d["exp_ctr"] = np.clip(w * loo + (1 - w) * d["global_ctr"], 1e-6, 1)
    d["exp_clicks"] = d["exp_ctr"] * d["impressions"]
    d["pearson"] = (d["clicks"] - d["exp_clicks"]) / np.sqrt(d["exp_clicks"])
    d.attrs["phi"] = float(np.median(d["pearson"] ** 2) / 0.4549)
    d["p_low"] = norm.cdf(d["pearson"] / np.sqrt(max(d.attrs["phi"], 1.0)))
    d["missed"] = (d["exp_clicks"] - d["clicks"]).clip(lower=0)
    d["flag"] = (d["p_low"] < 0.01) & (d["missed"] >= 5)
    d["ratio"] = (d["clicks"] + 0.5) / (d["exp_clicks"] + 0.5)
    return d

SC = {m: score(RAW[m]) for m in MONTHS}
for m in MONTHS:
    print(f"{m}: kept {len(SC[m]):,} pages | phi {SC[m].attrs['phi']:.1f} | flagged {SC[m]['flag'].sum():,} ({SC[m]['flag'].mean():.1%})")

OUT = []
for a, b in PAIRS:
    A, B = SC[a], SC[b]
    M = A.merge(B, on="content_hash_id", suffixes=("_a", "_b"))
    fa = M["flag_a"]
    extreme = (M["clicks_a"] < 0.05 * M["exp_clicks_a"]) & (M["exp_clicks_a"] >= 1000)
    fa_clean = fa & ~extreme
    rest = ~fa
    line = {}
    line["pair"] = f"{a} -> {b}"
    line["pages_in_both"] = len(M)
    line["flagged_in_A"] = int(fa.sum())
    line["flagged_in_A_not_extreme"] = int(fa_clean.sum())
    line["median_ratio_B_flagged"] = round(float(M.loc[fa, "ratio_b"].median()), 3)
    line["median_ratio_B_flagged_not_extreme"] = round(float(M.loc[fa_clean, "ratio_b"].median()), 3)
    line["median_ratio_B_rest"] = round(float(M.loc[rest, "ratio_b"].median()), 3)
    line["share_B_under_half_expected_flagged"] = round(float((M.loc[fa, "ratio_b"] < 0.5).mean()), 3)
    line["share_B_under_half_expected_rest"] = round(float((M.loc[rest, "ratio_b"] < 0.5).mean()), 3)
    line["flagged_again_in_B_flagged"] = round(float(M.loc[fa, "flag_b"].mean()), 3)
    line["flagged_in_B_rest"] = round(float(M.loc[rest, "flag_b"].mean()), 3)
    line["spearman_log_ratio_A_vs_B"] = round(float(spearmanr(np.log(M["ratio_a"]), np.log(M["ratio_b"]))[0]), 3)
    line["share_of_pages_flagged_A"] = round(float(fa.mean()), 3)
    line["share_of_B_missed_clicks_in_flagged_A"] = round(float(M.loc[fa, "missed_b"].sum() / M["missed_b"].sum()), 3)
    line["clients_with_flagged_A"] = int(M.loc[fa, "client_hash_id_a"].nunique())
    OUT.append(line)

print("\n===== COPY EVERYTHING BELOW THIS LINE AND SEND IT TO ME =====")
for line in OUT:
    print()
    for k, v in line.items():
        print(f"{k}: {v}")
#### END PART 2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-03 pages with impressions: 176,738 | clients: 47


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04 pages with impressions: 194,760 | clients: 51


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-05 pages with impressions: 237,910 | clients: 56

Position scale check (median of pos, median of sum_position/impressions, median difference): 8.18 8.18 0.0
2026-03: kept 101,441 pages | phi 2.0 | flagged 3,419 (3.4%)
2026-04: kept 107,144 pages | phi 1.8 | flagged 3,626 (3.4%)
2026-05: kept 112,309 pages | phi 1.7 | flagged 3,380 (3.0%)

===== COPY EVERYTHING BELOW THIS LINE AND SEND IT TO ME =====

pair: 2026-03 -> 2026-04
pages_in_both: 88482
flagged_in_A: 3413
flagged_in_A_not_extreme: 3413
median_ratio_B_flagged: 0.295
median_ratio_B_flagged_not_extreme: 0.295
median_ratio_B_rest: 0.718
share_B_under_half_expected_flagged: 0.754
share_B_under_half_expected_rest: 0.3
flagged_again_in_B_flagged: 0.518
flagged_in_B_rest: 0.02
spearman_log_ratio_A_vs_B: 0.49
share_of_pages_flagged_A: 0.039
share_of_B_missed_clicks_in_flagged_A: 0.36
clients_with_flagged_A: 22

pair: 2026-04 -> 2026-05
pages_in_both: 93247
flagged_in_A: 3611
flagged_in_A_not_extreme: 3611
median_ratio_B_flagged: 0

In [1]:
# CTR scoring notebook 6: does the A-ranking beat a size-only ranking at covering next month's shortfall?
# Paste this whole file into ONE Colab cell and press Run. Needs the Colab secret HF_TOKEN (Notebook access ON).
# It reads only three monthly files, aggregates them per page in DuckDB, and prints NO ids and NO client info.

#### PART 1: load (needs internet + token)
!pip install -q duckdb huggingface_hub
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from scipy.stats import norm, poisson, spearmanr

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{userdata.get('HF_TOKEN')}')")
BASE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

PAIRS = [("2026-03", "2026-04"), ("2026-04", "2026-05")]   # (window A = used to flag, window B = next month, used to check)
MONTHS = sorted({m for p in PAIRS for m in p})

def load_month(m):
    q = f'''
    SELECT client_hash_id, content_hash_id,
           sum(gsc_impressions) AS impressions, sum(gsc_clicks) AS clicks,
           sum(gsc_avg_position * gsc_impressions) / sum(gsc_impressions) AS pos,
           sum(gsc_sum_position) * 1.0 / sum(gsc_impressions) AS pos_from_sum
    FROM read_parquet('{BASE}/month={m}/data_0.parquet')
    WHERE gsc_impressions > 0
    GROUP BY 1, 2
    '''
    return con.sql(q).df()

RAW = {}
for m in MONTHS:
    RAW[m] = load_month(m)
    print(m, "pages with impressions:", f"{len(RAW[m]):,}", "| clients:", RAW[m]["client_hash_id"].nunique())
#### END PART 1

#### PART 2: analysis (uses RAW only)
BINS = [0, 1.5, 2.5, 3.5, 4.5, 5.5, 7.5, 10.5, 15.5, 20.5, 1000]
K = 50_000
MIN_IMP = 100

def score(raw):
    d = raw[(raw["impressions"] >= MIN_IMP) & raw["pos"].notna()].copy()
    d["ctr"] = d["clicks"] / d["impressions"]
    d["bucket"] = pd.cut(d["pos"], bins=BINS, labels=False)
    g = d.groupby("bucket")[["clicks", "impressions"]].sum()
    d["global_ctr"] = d["bucket"].map(g["clicks"] / g["impressions"])
    cb = d.groupby(["client_hash_id", "bucket"])[["clicks", "impressions"]].transform("sum")
    li, lc = cb["impressions"] - d["impressions"], cb["clicks"] - d["clicks"]
    loo = np.where(li > 0, lc / li.where(li > 0, 1), 0.0)
    w = li / (li + K)
    d["exp_ctr"] = np.clip(w * loo + (1 - w) * d["global_ctr"], 1e-6, 1)
    d["exp_clicks"] = d["exp_ctr"] * d["impressions"]
    d["pearson"] = (d["clicks"] - d["exp_clicks"]) / np.sqrt(d["exp_clicks"])
    phi = float(np.median(d["pearson"] ** 2) / 0.4549)
    d["p_low"] = norm.cdf(d["pearson"] / np.sqrt(max(phi, 1.0)))
    d["missed"] = (d["exp_clicks"] - d["clicks"]).clip(lower=0)
    d["flag"] = (d["p_low"] < 0.01) & (d["missed"] >= 5)
    d["ratio"] = (d["clicks"] + 0.5) / (d["exp_clicks"] + 0.5)
    return d


SC = {m: score(RAW[m]) for m in MONTHS}
OUT = []
for a, b in PAIRS:
    M = SC[a].merge(SC[b], on="content_hash_id", suffixes=("_a", "_b"))
    tot = M["missed_b"].sum()
    n = len(M)
    ranks = {
        "A: biggest estimated missed clicks (our ranking)": M.sort_values("missed_a", ascending=False),
        "size only: most impressions": M.sort_values("impressions_a", ascending=False),
        "gap only: lowest clicks/expected ratio": M.sort_values("ratio_a", ascending=True),
    }
    line = {"pair": f"{a} -> {b}"}
    for name, R in ranks.items():
        line[name] = {f"top {int(q*100)}%": round(float(R["missed_b"].head(int(n * q)).sum() / tot), 3) for q in [0.01, 0.02, 0.05, 0.10]}
    OUT.append(line)

print("===== COPY EVERYTHING BELOW THIS LINE AND SEND IT TO ME =====")
for line in OUT:
    print()
    for k, v in line.items():
        print(f"{k}: {v}")
#### END PART 2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-03 pages with impressions: 176,738 | clients: 47


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04 pages with impressions: 194,760 | clients: 51


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-05 pages with impressions: 237,910 | clients: 56
===== COPY EVERYTHING BELOW THIS LINE AND SEND IT TO ME =====

pair: 2026-03 -> 2026-04
A: biggest estimated missed clicks (our ranking): {'top 1%': 0.203, 'top 2%': 0.286, 'top 5%': 0.43, 'top 10%': 0.56}
size only: most impressions: {'top 1%': 0.147, 'top 2%': 0.225, 'top 5%': 0.367, 'top 10%': 0.508}
gap only: lowest clicks/expected ratio: {'top 1%': 0.05, 'top 2%': 0.079, 'top 5%': 0.138, 'top 10%': 0.213}

pair: 2026-04 -> 2026-05
A: biggest estimated missed clicks (our ranking): {'top 1%': 0.264, 'top 2%': 0.347, 'top 5%': 0.482, 'top 10%': 0.587}
size only: most impressions: {'top 1%': 0.237, 'top 2%': 0.333, 'top 5%': 0.471, 'top 10%': 0.612}
gap only: lowest clicks/expected ratio: {'top 1%': 0.041, 'top 2%': 0.063, 'top 5%': 0.12, 'top 10%': 0.182}
